# automating ROI PC

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "histology_pipeline.py").is_file())
DATA_ROOT = Path(os.environ.get("PROSTATE_DATA_ROOT", str(REPO_ROOT / "data"))).expanduser().resolve()
EXAMPLE_IMAGE_ID = os.environ.get("PANDA_IMAGE_ID", "SET_YOUR_AUTHORISED_IMAGE_ID")


# understanding the dataset

In [ ]:
import openslide
from PIL import Image
import matplotlib.pyplot as plt
import os

In [ ]:
# Set paths
image_dir = f'{DATA_ROOT}/panda/train_images'
mask_dir = f'{DATA_ROOT}/panda/train_label_masks'

# List files
image_files = sorted(os.listdir(image_dir))
mask_files = sorted(os.listdir(mask_dir))

# Pick the first file (they should have the same filenames)
sample_image_path = os.path.join(image_dir, image_files[0])
sample_mask_path = os.path.join(mask_dir, mask_files[0])

In [ ]:
# Load image and mask
slide = openslide.OpenSlide(sample_image_path)
mask = openslide.OpenSlide(sample_mask_path)

# Read region (e.g., top-left 1024x1024 pixels at level 0)
img_patch = slide.read_region((0, 0), 0, (1024, 1024)).convert("RGB")
mask_patch = mask.read_region((0, 0), 0, (1024, 1024)).convert("RGB")

# Display both side-by-side
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img_patch)
axs[0].set_title('Histopathology Image')
axs[0].axis('off')

axs[1].imshow(mask_patch)
axs[1].set_title('Label Mask')
axs[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Use level 2 to get downsampled image
level = 2  # usually safe for overview

# Read entire downsampled slide
slide_img = slide.read_region((0, 0), level, slide.level_dimensions[level]).convert("RGB")

plt.figure(figsize=(10, 10))
plt.imshow(slide_img)
plt.title("Whole Slide (Level 2)")
plt.axis("off")
plt.show()


In [ ]:
x, y = 20000, 30000  # or whatever coordinates you used that worked

In [ ]:
import numpy as np

# Convert mask to grayscale
mask_patch = mask.read_region((x, y), 0, (1024, 1024)).convert("L")

# Convert to NumPy array
mask_np = np.array(mask_patch)

# Visualize
plt.imshow(mask_np, cmap='gray')
plt.title("Mask Patch")
plt.axis("off")
plt.show()

# Print unique values
print("Unique pixel values in mask:", np.unique(mask_np))


In [ ]:
# Check available levels
print("Mask levels:", mask.level_count)
print("Level 0 dimensions:", mask.level_dimensions[0])
for i in range(mask.level_count):
    print(f"Level {i}: {mask.level_dimensions[i]}")

# Choose a lower level (e.g., level 2)
level = 2
dims = mask.level_dimensions[level]

# Read the full mask at that level
mask_lowres = mask.read_region((0, 0), level, dims).convert("L")

# Convert to NumPy array and display
mask_array = np.array(mask_lowres)

plt.figure(figsize=(10, 10))
plt.imshow(mask_array, cmap='gray')
plt.title(f"Full Mask at Level {level}")
plt.axis('off')
plt.show()

# Show unique pixel values (class labels)
print("Unique values in mask:", np.unique(mask_array))


In [ ]:
from tqdm import tqdm

image_dir = f'{DATA_ROOT}/panda/train_images'
mask_dir = f'{DATA_ROOT}/panda/train_label_masks'

image_files = sorted(os.listdir(image_dir))
mask_files = sorted(os.listdir(mask_dir))

for img_file, mask_file in tqdm(zip(image_files, mask_files), total=len(image_files)):
    mask_path = os.path.join(mask_dir, mask_file)
    try:
        m = openslide.OpenSlide(mask_path)
        level = 2
        dims = m.level_dimensions[level]
        mask_img = m.read_region((0, 0), level, dims).convert("L")
        mask_np = np.array(mask_img)
        unique_vals = np.unique(mask_np)

        if len(unique_vals) > 1 or unique_vals[0] != 0:
            print(f"✅ Found non-empty mask: {mask_file}, Unique values: {unique_vals}")
            break

    except Exception as e:
        print(f"Error in {mask_file}: {e}")


In [ ]:
image_id = f"{EXAMPLE_IMAGE_ID}"
image_path = f"{DATA_ROOT}/panda/train_images/{image_id}.tiff"
mask_path = f"{DATA_ROOT}/panda/train_label_masks/{image_id}_mask.tiff"

In [ ]:
slide = openslide.OpenSlide(image_path)
mask = openslide.OpenSlide(mask_path)

# Read both at low resolution (level 2)
level = 2
dims = slide.level_dimensions[level]

slide_img = slide.read_region((0, 0), level, dims).convert("RGB")
mask_img = mask.read_region((0, 0), level, dims).convert("L")

# Plot side by side
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(16, 8))

plt.subplot(1, 2, 1)
plt.imshow(slide_img)
plt.title("Histopathology Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask_img, cmap='gray')
plt.title("Label Mask")
plt.axis("off")

plt.show()

# Optional: Check unique mask values
print("Mask labels:", np.unique(np.array(mask_img)))


In [ ]:
import pandas as pd

csv_path = f"{DATA_ROOT}/panda/train.csv"
df = pd.read_csv(csv_path)

# Show first few rows
df.head()

In [ ]:
# Load CSV if not already loaded
import pandas as pd

csv_path = f"{DATA_ROOT}/panda/train.csv"
df = pd.read_csv(csv_path)

# Group by data_provider and get unique ISUP grades
isup_by_provider = df.groupby('data_provider')['isup_grade'].unique()

# Convert to a dictionary for cleaner printing
isup_dict = isup_by_provider.to_dict()

# Group by data_provider and get unique ISUP grades
gleason_by_provider = df.groupby('data_provider')['gleason_score'].unique()

# Convert to a dictionary for cleaner printing
gleason_dict = gleason_by_provider.to_dict()

# Print the results
for provider, grades in isup_dict.items():
    print(f"{provider}: {sorted(grades)}")

# Print the results
for provider, grades in gleason_dict.items():
    print(f"{provider}: {sorted(grades)}")

In [ ]:
import os

image_dir = f'{DATA_ROOT}/panda/train_images'
mask_dir = f'{DATA_ROOT}/panda/train_label_masks'

image_files = [f for f in os.listdir(image_dir) if f.endswith('.tiff')]
mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.tiff')]

print("Number of image files:", len(image_files))
print("Number of mask files:", len(mask_files))


In [ ]:
import pandas as pd

csv_path = f'{DATA_ROOT}/panda/train.csv'
df = pd.read_csv(csv_path)

print("Number of rows in train.csv:", len(df))

In [ ]:
# Extract IDs from filenames
image_ids = set(f.replace('.tiff', '') for f in image_files)
mask_ids = set(f.replace('_mask.tiff', '') for f in mask_files)

# Find missing mask IDs
missing_mask_ids = sorted(list(image_ids - mask_ids))

print("Slides missing masks:", len(missing_mask_ids))
print(missing_mask_ids[:10])  # show first 10

In [ ]:
# Group by data_provider and isup_grade
grouped = df.groupby(['data_provider', 'isup_grade'])['gleason_score'].unique()

# Print results
for (provider, grade), gleasons in grouped.items():
    print(f"{provider} — ISUP {grade}: {sorted(gleasons)}")


In [ ]:
import openslide

# Load the slide
slide = openslide.OpenSlide(f"{DATA_ROOT}/panda/train_images/{image_id}.tiff")

# Get number of levels
print("Number of resolution levels:", slide.level_count)

# Get dimensions at each level
for i in range(slide.level_count):
    print(f"Level {i} dimensions: {slide.level_dimensions[i]}")

In [ ]:
import tifffile

In [ ]:
tiff_path = f"{DATA_ROOT}/panda/train_images/{image_id}.tiff"

with tifffile.TiffFile(tiff_path) as tif:
    for page in tif.pages:
        print("Tags for page:", page.index)
        for tag in page.tags.values():
            print(f"{tag.name}: {tag.value}")


# preprocessing the image

In [ ]:
import openslide
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

In [ ]:
import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from cellpose import models

def segment_and_mark_cells(patch, display=True, diameter_value=8):

    # Load the Cellpose model (pre-trained model, using 'cyto' mode for general cell segmentation)
    model = models.Cellpose(gpu=False, model_type='cyto')  # Use GPU=True if you have a GPU available

    # Convert the patch to grayscale if necessary
    if patch.shape[-1] == 3:
        patch_gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    else:
        patch_gray = patch

    # Use the Cellpose model to segment the patch with the specified smaller diameter
    masks, flows, styles, diams = model.eval(patch_gray, diameter=diameter_value, channels=[0, 0])

    # Create an RGB version of the original patch to draw the individual cell annotations
    marked_patch = patch.copy()

    # Iterate over each unique cell ID in the mask and mark it
    unique_cells = np.unique(masks)

    for cell_id in unique_cells:
        if cell_id == 0:
            continue  # Skip background

        # Create a mask for the current cell
        cell_mask = (masks == cell_id)

        # Find contours of the cell
        contours, _ = cv2.findContours(cell_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Draw the contour on the patch without displaying the cell number
        cv2.drawContours(marked_patch, contours, -1, (0, 255, 0), 2)  # Draw the cell contour in green

    # Display the original patch and the patch with individually marked cells
    if display:
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1), plt.title("Original Patch"), plt.imshow(patch), plt.axis('off')
        plt.subplot(1, 2, 2), plt.title("Individually Marked Cells"), plt.imshow(marked_patch), plt.axis('off')
        plt.show()

    return marked_patch

def process_all_patches_with_individual_marking(image_path, patch_size=256, white_pixel_threshold=0.2, diameter_value=8):

    try:
        slide = openslide.OpenSlide(image_path)
    except openslide.OpenSlideError as e:
        print(f"Error opening slide: {e}")
        return

    for zoom_level in range(len(slide.level_downsamples)):
        width, height = slide.level_dimensions[zoom_level]
        print(f"Processing zoom level {zoom_level} with dimensions: {width}x{height}")

        # Read the image at the current zoom level
        slide_image = slide.read_region((0, 0), zoom_level, (width, height))
        slide_image = slide_image.convert("RGB")
        slide_image_np = np.array(slide_image)

        step_size = patch_size // 2  # 50% overlap
        displayed_patches = 0

        for y in range(0, height - patch_size + 1, step_size):
            for x in range(0, width - patch_size + 1, step_size):
                patch = slide_image_np[y:y + patch_size, x:x + patch_size]

                # Check the percentage of white pixels
                white_pixels = np.all(patch > 200, axis=2)
                white_pixel_ratio = np.sum(white_pixels) / (patch_size * patch_size)

                # Select patches with a white pixel ratio less than 0.2
                if white_pixel_ratio < white_pixel_threshold:
                    displayed_patches += 1

                    # Segment and mark each cell individually using Cellpose
                    marked_patch = segment_and_mark_cells(patch, display=True, diameter_value=diameter_value)

                    # Save the marked patch if needed
                    patch_filename = f'cellpose_marked_zoom_{zoom_level}_patch_{x}_{y}.png'
                    # Image.fromarray(marked_patch).save(patch_filename)
                    print(f"Marked patch saved: {patch_filename}")

        print(f"Total patches processed at zoom level {zoom_level}: {displayed_patches}")

# Example usage
image_path = f'{DATA_ROOT}/automation/input/{EXAMPLE_IMAGE_ID}.tiff'
process_all_patches_with_individual_marking(image_path, patch_size=256, white_pixel_threshold=0.2, diameter_value=12)


In [ ]:
# Set the desired smaller diameter value
smaller_diameter = 10

# Use the Cellpose model to segment the patch with the specified diameter
masks, flows, styles, diams = model.eval(patch_gray, diameter=smaller_diameter, channels=[0, 0])  # Set diameter to 10 pixels


In [ ]:
def segment_and_mark_cells(patch, display=True, diameter_value=10):
    """
    Segment and mark each cell individually in a patch using the Cellpose model with a specified diameter.

    Args:
        patch: A NumPy array representing the extracted patch.
        display: Whether to display the segmentation result (default: True).
        diameter_value: The diameter to be used by the Cellpose model for segmentation (default: 10 pixels).

    Returns:
        The patch with individually marked cells.
    """
    # Load the Cellpose model (pre-trained model, using 'cyto' mode for general cell segmentation)
    model = models.Cellpose(gpu=False, model_type='cyto')  # Use GPU=True if you have a GPU available

    # Convert the patch to grayscale if necessary
    if patch.shape[-1] == 3:
        patch_gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    else:
        patch_gray = patch

    # Use the Cellpose model to segment the patch with the specified diameter
    masks, flows, styles, diams = model.eval(patch_gray, diameter=diameter_value, channels=[0, 0])

    # The rest of the code remains the same for marking cells
    marked_patch = patch.copy()
    unique_cells = np.unique(masks)

    for cell_id in unique_cells:
        if cell_id == 0:
            continue  # Skip background

        cell_mask = (masks == cell_id)
        contours, _ = cv2.findContours(cell_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(marked_patch, contours, -1, (0, 255, 0), 2)
        x, y, w, h = cv2.boundingRect(contours[0])
        cv2.putText(marked_patch, str(cell_id), (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1, cv2.LINE_AA)

    if display:
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1), plt.title("Original Patch"), plt.imshow(patch), plt.axis('off')
        plt.subplot(1, 2, 2), plt.title("Individually Marked Cells"), plt.imshow(marked_patch), plt.axis('off')
        plt.show()

    return marked_patch


In [ ]:
def segment_and_mark_cells(patch, display=True, diameter_value=10):
    """
    Segment and mark each cell individually in a patch using the Cellpose model with a specified diameter.

    Args:
        patch: A NumPy array representing the extracted patch.
        display: Whether to display the segmentation result (default: True).
        diameter_value: The diameter to be used by the Cellpose model for segmentation (default: 10 pixels).

    Returns:
        The patch with individually marked cells.
    """
    # Load the Cellpose model (pre-trained model, using 'cyto' mode for general cell segmentation)
    model = models.Cellpose(gpu=False, model_type='cyto')  # Use GPU=True if you have a GPU available

    # Convert the patch to grayscale if necessary
    if patch.shape[-1] == 3:
        patch_gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    else:
        patch_gray = patch

    # Use the Cellpose model to segment the patch with the specified diameter
    masks, flows, styles, diams = model.eval(patch_gray, diameter=diameter_value, channels=[0, 0])

    # The rest of the code remains the same for marking cells
    marked_patch = patch.copy()
    unique_cells = np.unique(masks)

    for cell_id in unique_cells:
        if cell_id == 0:
            continue  # Skip background

        cell_mask = (masks == cell_id)
        contours, _ = cv2.findContours(cell_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(marked_patch, contours, -1, (0, 255, 0), 2)
        x, y, w, h = cv2.boundingRect(contours[0])
        cv2.putText(marked_patch, str(cell_id), (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1, cv2.LINE_AA)

    if display:
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1), plt.title("Original Patch"), plt.imshow(patch), plt.axis('off')
        plt.subplot(1, 2, 2), plt.title("Individually Marked Cells"), plt.imshow(marked_patch), plt.axis('off')
        plt.show()

    return marked_patch


In [ ]:
# Example usage with a smaller diameter value
marked_patch = segment_and_mark_cells(patch_np, display=True, diameter_value=8)  # Set diameter to 8 pixels


In [ ]:
import svgwrite
import numpy as np
import cv2
from cellpose import models

def create_svg_for_cell(cell_contour, cell_id, output_dir="output_svgs"):
    """
    Creates an SVG file for a single cell contour.

    Args:
        cell_contour: Contour points of the cell.
        cell_id: Unique ID for the cell.
        output_dir: Directory where the SVG file will be saved.
    """
    # Create the SVG drawing
    dwg = svgwrite.Drawing(filename=f"{output_dir}/cell_{cell_id}.svg", size=('256px', '256px'))

    # Create a path string from the contour points
    path_str = "M " + " L ".join([f"{pt[0][0]},{pt[0][1]}" for pt in cell_contour]) + " Z"

    # Add the path to the SVG drawing
    dwg.add(dwg.path(d=path_str, fill='none', stroke='black', stroke_width=1))

    # Save the SVG
    dwg.save()
    print(f"SVG file created for cell {cell_id}: {output_dir}/cell_{cell_id}.svg")

def create_combined_svg(cell_contours, output_dir="output_svgs"):
    """
    Creates a single SVG file that combines all cell contours.

    Args:
        cell_contours: List of contours for all cells.
        output_dir: Directory where the SVG file will be saved.
    """
    # Create the SVG drawing
    #dwg = svgwrite.Drawing(filename=f"{output_dir}/combined_cells.svg", size=('256px', '256px'))

    # Add all contours as paths to the SVG
    for cell_contour in cell_contours:
        path_str = "M " + " L ".join([f"{pt[0][0]},{pt[0][1]}" for pt in cell_contour]) + " Z"
        #dwg.add(dwg.path(d=path_str, fill='none', stroke='black', stroke_width=1))

    # Save the SVG
    #dwg.save()
    print(f"Combined SVG file created: {output_dir}/combined_cells.svg")

def generate_svgs_from_patch(patch):
    """
    Generates individual SVGs for each cell and a combined SVG for the given patch.

    Args:
        patch: NumPy array representing the patch image.
    """
    # Load the Cellpose model
    model = models.Cellpose(gpu=False, model_type='cyto')

    # Convert to grayscale if needed
    if patch.shape[-1] == 3:
        patch_gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
    else:
        patch_gray = patch

    # Perform segmentation
    masks, flows, styles, diams = model.eval(patch_gray, diameter=None, channels=[0, 0])

    # Extract contours for each cell
    cell_contours = []
    unique_cells = np.unique(masks)

    for cell_id in unique_cells:
        if cell_id == 0:
            continue  # Skip background

        # Create a mask for the current cell
        cell_mask = (masks == cell_id)

        # Find the contour for the current cell
        contours, _ = cv2.findContours(cell_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            cell_contours.append(contours[0])  # Only consider the largest contour if multiple are found
            create_svg_for_cell(contours[0], cell_id)  # Create an individual SVG

    # Create a combined SVG for all cells
    create_combined_svg(cell_contours)

# Example usage
# Assuming `patch_np` is the NumPy array of your patch
generate_svgs_from_patch(patch_np)


In [ ]:
import openslide
import numpy as np

# Define the path to your WSI (whole-slide image)
image_path = f'{DATA_ROOT}/automation/input/{EXAMPLE_IMAGE_ID}.tiff'

# Open the WSI using OpenSlide
slide = openslide.OpenSlide(image_path)

# Define the level and region to extract (e.g., at zoom level 0, you get the highest resolution)
zoom_level = 0  # The highest resolution level
width, height = slide.level_dimensions[zoom_level]
print(f"Image dimensions at zoom level {zoom_level}: {width}x{height}")

# Extract a patch from the image at the specified level
x_coord, y_coord = 1000, 1000  # Define the top-left corner coordinates of the patch
patch_size = 256  # Define the patch size

# Read the patch from the WSI
patch = slide.read_region((x_coord, y_coord), zoom_level, (patch_size, patch_size))
patch = patch.convert("RGB")  # Convert to RGB format
patch_np = np.array(patch)  # Convert to a NumPy array

# Display the patch to confirm
import matplotlib.pyplot as plt
plt.imshow(patch_np)
plt.title(f"Extracted Patch at ({x_coord}, {y_coord})")
plt.axis('off')
plt.show()


In [ ]:
# Generate SVGs from the extracted patch
generate_svgs_from_patch(patch_np)


# visualizing the labelled mask present in PANDA dataset

In [ ]:
import openslide
import matplotlib.pyplot as plt

# Specify the path to the TIFF file
image_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'

# Load the image using OpenSlide
slide = openslide.OpenSlide(image_path)

# Print the dimensions of the image
dimensions = slide.dimensions
print("Image Dimensions:", dimensions)

# Get the number of zoom levels
num_levels = slide.level_count
print("Number of Zoom Levels:", num_levels)

# Print the dimensions at each zoom level
for i in range(num_levels):
    print(f"Zoom level {i}: {slide.level_dimensions[i]}")

# Let's downsample the image by using a higher zoom level (e.g., level 2)
downsample_level = 0  # Adjust this to downsample more or less
downsampled_region = slide.read_region((0, 0), downsample_level, slide.level_dimensions[downsample_level])

# Convert the downsampled region to RGB
downsampled_region_rgb = downsampled_region.convert("RGB")

# Display the downsampled image
plt.figure(figsize=(10, 10))
plt.imshow(downsampled_region_rgb)
plt.title(f'Prostate Cancer Mask Image - Downsampled (Zoom Level {downsample_level})')
plt.axis('off')
plt.show()


In [ ]:
import openslide
import matplotlib.pyplot as plt

# Path to the WSI
wsi_path = f'{DATA_ROOT}/panda/train_images/{EXAMPLE_IMAGE_ID}.tiff'

# Path to the corresponding mask
mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'

# Open the WSI and mask using OpenSlide
wsi = openslide.OpenSlide(wsi_path)
mask = openslide.OpenSlide(mask_path)

print(wsi)
print(mask)
# Select the lowest zoom level (highest number) to view the entire image
# You can choose the lowest zoom level to make the image smaller and viewable
wsi_zoom_level = wsi.level_count - 1
mask_zoom_level = mask.level_count - 1

# Read the entire region at the selected zoom level
wsi_region = wsi.read_region((0, 0), wsi_zoom_level, wsi.level_dimensions[wsi_zoom_level])
mask_region = mask.read_region((0, 0), mask_zoom_level, mask.level_dimensions[mask_zoom_level])

# Convert both regions to RGB format for proper display
wsi_region = wsi_region.convert("RGB")
mask_region = mask_region.convert("RGB")

# Display the WSI and mask side by side
plt.figure(figsize=(15, 10))
plt.subplot(1, 2, 1)
plt.imshow(wsi_region)
plt.title('WSI Full Image (Zoom Level {})'.format(wsi_zoom_level))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(mask_region)
plt.title('Mask Full Image (Zoom Level {})'.format(mask_zoom_level))
plt.axis('off')

plt.show()

# Close the OpenSlide objects to free up resources
wsi.close()
mask.close()


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Increase the maximum allowed image pixels to handle large images
Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check

# Path to the corresponding mask
mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'

# Open the mask using PIL
mask = Image.open(mask_path)

# Display the mask
plt.figure(figsize=(10, 10))
plt.imshow(mask)
plt.title('Mask Image')
plt.axis('off')
plt.show()


In [ ]:
import numpy as np
import openslide
import matplotlib.pyplot as plt

# Path to the mask file
mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'

# Open the mask using OpenSlide
mask_slide = openslide.OpenSlide(mask_path)

# Read the entire mask region at the highest resolution (level 0)
mask_region = mask_slide.read_region((0, 0), 0, mask_slide.dimensions)

# Convert the mask region to grayscale (if it's RGB)
mask_region_np = np.array(mask_region.convert('L'))

# Display the mask
plt.imshow(mask_region_np, cmap='gray')
plt.title('Mask (Grayscale)')
plt.colorbar(label="Pixel Value (Gleason Grade)")
plt.show()

# Inspect pixel values
unique_values = np.unique(mask_region_np)
print("Unique pixel values in the mask:", unique_values)


In [ ]:
import numpy as np
from PIL import Image

# Increase the maximum allowed image pixels to handle large images
Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check


# Open the mask using PIL
mask = Image.open(mask_path)

# Convert mask to a NumPy array for analysis
mask_np = np.array(mask)

# Print unique pixel values in the mask
unique_values = np.unique(mask_np)
print("Unique pixel values in the mask:", unique_values)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Path to the mask
mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'


# Disable decompression bomb warning
Image.MAX_IMAGE_PIXELS = None


# Load the mask
mask = Image.open(mask_path)
mask_np = np.array(mask)

# Visualize the regions for Gleason 3, 4, and 5
gleason_3 = mask_np == 3  # Mask for Gleason 3
gleason_4 = mask_np == 4  # Mask for Gleason 4
gleason_5 = mask_np == 5  # Mask for Gleason 5

# Plot the masks
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(gleason_3, cmap='Reds')
plt.title('Gleason 3')
plt.subplot(1, 3, 2)
plt.imshow(gleason_4, cmap='Blues')
plt.title('Gleason 4')
plt.subplot(1, 3, 3)
plt.imshow(gleason_5, cmap='Greens')
plt.title('Gleason 5')
plt.show()


In [ ]:
import pandas as pd

# Path to the CSV file
csv_path = f'{DATA_ROOT}/panda/train.csv'

# Read the CSV file using pandas
df = pd.read_csv(csv_path)

# Display the contents of the CSV file
print("CSV file contents:")

print(df)


In [ ]:
import glob

# Path to the folder with a pattern to match .tiff files
file_pattern = f'{DATA_ROOT}/panda/train_label_masks/*.tiff'

# Get the list of all .tiff files in the folder
file_list = glob.glob(file_pattern)

# Count the number of files
num_files = len(file_list)

# Print the number of files
print(f"Number of TIFF files in the folder: {num_files}")


In [ ]:
import glob

# Path to the folder with a pattern to match .tiff files
file_pattern_images = f'{DATA_ROOT}/panda/train_images/*.tiff'

# Get the list of all .tiff files in the folder
file_list_images = glob.glob(file_pattern_images)

# Count the number of files
num_files_images = len(file_list_images)

# Print the number of files
print(f"Number of TIFF files in the folder: {num_files_images}")


In [ ]:
import numpy as np
from PIL import Image

# Increase the maximum allowed image pixels to handle large images
Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check

mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'


# Open the mask using PIL
mask = Image.open(mask_path)

# Convert mask to a NumPy array for analysis
mask_np = np.array(mask)

# Print unique pixel values in the mask
unique_values = np.unique(mask_np)
print("Unique pixel values in the mask:", unique_values)

In [ ]:
mask_np.shape

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Increase the maximum allowed image pixels to handle large images
Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check

mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'

# Open the mask using PIL
mask = Image.open(mask_path)

# Convert mask to a NumPy array for analysis
mask_np = np.array(mask)

# Check if the mask has 3 channels (RGB)
if mask_np.ndim == 3 and mask_np.shape[2] == 3:
    # Split the channels: Red, Green, Blue
    red_channel = mask_np[:, :, 0]
    green_channel = mask_np[:, :, 1]
    blue_channel = mask_np[:, :, 2]

    # Display the three channels separately
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(red_channel, cmap='gray')
    plt.title('Red Channel')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(green_channel, cmap='gray')
    plt.title('Green Channel')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(blue_channel, cmap='gray')
    plt.title('Blue Channel')
    plt.axis('off')

    plt.show()

    # Print unique values in each channel to understand the content
    print("Unique values in Red Channel:", np.unique(red_channel))
    print("Unique values in Green Channel:", np.unique(green_channel))
    print("Unique values in Blue Channel:", np.unique(blue_channel))

else:
    print("The mask does not have 3 channels.")


In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Increase the maximum allowed image pixels to handle large images
Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check

mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'

# Open the mask using PIL
mask = Image.open(mask_path)

# Convert mask to a NumPy array for analysis
mask_np = np.array(mask)

# Extract the red channel (contains the mask information)
red_channel = mask_np[:, :, 0]

# Define the color mapping for unique values
color_mapping = {
    0: [255, 255, 255],  # White for background
    1: [0, 255, 0],      # Green for benign tissue
    2: [255, 0, 0]       # Red for cancerous tissue
}

# Create a new RGB image where each unique value is color-coded
color_mask = np.zeros((red_channel.shape[0], red_channel.shape[1], 3), dtype=np.uint8)

# Apply the color mapping
for value, color in color_mapping.items():
    color_mask[red_channel == value] = color

# Display the color-coded mask
plt.figure(figsize=(10, 10))
plt.imshow(color_mask)
plt.title('Color-Coded Mask')
plt.axis('off')
plt.show()


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Define the mask path and folders
mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'
image_folder = f'{DATA_ROOT}/panda/train_images'

# Step 1: Extract the image ID from the mask file name
image_id = os.path.basename(mask_path).split('_')[0]

# Step 2: Construct the path to the corresponding image in the train_images folder
image_path = os.path.join(image_folder, f'{image_id}.tiff')

# Increase the maximum allowed image pixels to handle large images
Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check

# Step 3: Open the corresponding image and mask
image = Image.open(image_path)
mask = Image.open(mask_path)

# Convert mask to NumPy array for further processing
mask_np = np.array(mask)

# Step 4: Create a color-coded mask based on unique values in the mask
# Define color mapping for each unique value in the mask
color_mapping = {
    0: [255, 255, 255],  # White for background/unknown
    1: [0, 255, 0],      # Green for stroma (connective tissue)
    2: [255, 0, 0],      # Red for benign epithelium
    3: [0, 0, 255],      # Blue for cancerous epithelium (Gleason 3)
    4: [255, 255, 0],    # Yellow for cancerous epithelium (Gleason 4)
    5: [255, 0, 255]     # Magenta for cancerous epithelium (Gleason 5)
}

# Create an RGB image to store the color-coded mask
color_mask = np.zeros((mask_np.shape[0], mask_np.shape[1], 3), dtype=np.uint8)

# Map each unique value to its respective color
for value, color in color_mapping.items():
    color_mask[mask_np[:, :, 0] == value] = color  # Use the first channel for mapping

# Step 5: Display both the image and color-coded mask side by side
plt.figure(figsize=(30, 30))

# Display the original image
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title(f'Image: {image_id}')
plt.axis('off')

# Display the color-coded mask
plt.subplot(1, 2, 2)
plt.imshow(color_mask)
plt.title(f'Color-Coded Mask: {image_id}')
plt.axis('off')

plt.show()


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Define the mask path and folders
mask_path = f'{DATA_ROOT}/panda/train_label_masks/{EXAMPLE_IMAGE_ID}_mask.tiff'
image_folder = f'{DATA_ROOT}/panda/train_images'

# Step 1: Extract the image ID from the mask file name
image_id = os.path.basename(mask_path).split('_')[0]

# Step 2: Construct the path to the corresponding image in the train_images folder
image_path = os.path.join(image_folder, f'{image_id}.tiff')

# Increase the maximum allowed image pixels to handle large images
Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check

# Step 3: Open the corresponding image and mask
image = Image.open(image_path)
mask = Image.open(mask_path)

# Convert mask to NumPy array for further processing
mask_np = np.array(mask)

# Step 4: Create a color-coded mask based on unique values in the mask
# Define color mapping for each unique value in the mask
color_mapping = {
    0: [255, 255, 255],  # White for background/unknown
    1: [0, 255, 0],      # Dark green for stroma (connective tissue)
    2: [255, 192, 203],  # Light pink for benign epithelium
    3: [255, 0, 0],      # Red for cancerous epithelium (Gleason 3)
    4: [200, 0, 0],      # Darker red for cancerous epithelium (Gleason 4)
    5: [150, 0, 0]       # Even darker red for cancerous epithelium (Gleason 5)
}

# Create an RGB image to store the color-coded mask
color_mask = np.zeros((mask_np.shape[0], mask_np.shape[1], 3), dtype=np.uint8)

# Map each unique value to its respective color
for value, color in color_mapping.items():
    color_mask[mask_np[:, :, 0] == value] = color  # Use the first channel for mapping

# Step 5: Display both the image and color-coded mask side by side
plt.figure(figsize=(30, 30))

# Display the original image
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title(f'Image: {image_id}')
plt.axis('off')

# Display the color-coded mask
plt.subplot(1, 2, 2)
plt.imshow(color_mask)
plt.title(f'Color-Coded Mask: {image_id}')
plt.axis('off')

plt.show()


# color coding based on the data source

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Paths
train_csv_path = f'{DATA_ROOT}/panda/train.csv'
train_images_folder = f'{DATA_ROOT}/panda/train_images'
train_masks_folder = f'{DATA_ROOT}/panda/train_label_masks'

In [ ]:
# Step 1: Read the CSV file
train_df = pd.read_csv(train_csv_path)

In [ ]:
# Step 2: Filter for each data provider
radboud_df = train_df[train_df['data_provider'] == 'radboud']
karolinska_df = train_df[train_df['data_provider'] == 'karolinska']

# Step 3: Randomly select one image from each data provider
random_radboud = radboud_df.sample(1).iloc[0]
random_karolinska = karolinska_df.sample(1).iloc[0]

In [ ]:
random_radboud

In [ ]:
random_karolinska

In [ ]:
# Function to process and display the image and mask
def display_image_and_mask(image_id, data_provider):
    # Construct paths to image and mask
    image_path = os.path.join(train_images_folder, f'{image_id}.tiff')
    mask_path = os.path.join(train_masks_folder, f'{image_id}_mask.tiff')

    # Increase the maximum allowed image pixels to handle large images
    Image.MAX_IMAGE_PIXELS = None  # Disable the decompression bomb check
    # Check if both the image and mask exist
    if os.path.exists(image_path) and os.path.exists(mask_path):
        # Open the image and mask
        image = Image.open(image_path)
        mask = Image.open(mask_path)

        # Convert mask to NumPy array for further processing
        mask_np = np.array(mask)

        # Apply color coding based on the `data_provider`
        if data_provider == 'radboud':
            color_mapping = {
                  0: [255, 255, 255],  # White for background/unknown
                  1: [0, 255, 0],      # Dark green for stroma (connective tissue)
                  2: [0, 255, 0],  # Light pink for benign epithelium
                  3: [255, 0, 0],
                  4: [255, 0, 0],
                  5: [255, 0, 0],
            }
        elif data_provider == 'karolinska':
            color_mapping = {
                0: [255, 255, 255],  # White for background/unknown
                1: [0, 255, 0],      # Green for benign tissue (stroma and epithelium combined)
                2: [255, 0, 0]       # Red for cancerous tissue (stroma and epithelium combined)
            }
        else:
            raise ValueError(f"Unknown data provider: {data_provider}")

        # Create a color-coded mask
        color_mask = np.zeros((mask_np.shape[0], mask_np.shape[1], 3), dtype=np.uint8)
        for value, color in color_mapping.items():
            color_mask[mask_np[:, :, 0] == value] = color  # Use the first channel for mapping

        # Display the image and color-coded mask side by side
        plt.figure(figsize=(15, 15))

        # Display the original image
        plt.subplot(1, 2, 1)
        plt.imshow(image)
        plt.title(f'Image: {image_id}')
        plt.axis('off')

        # Display the color-coded mask
        plt.subplot(1, 2, 2)
        plt.imshow(color_mask)
        plt.title(f'Color-Coded Mask: {image_id} ({data_provider})')
        plt.axis('off')

        plt.show()
    else:
        print(f"Image or mask for {image_id} not found!")

# Step 4: Display one random image from each data provider
print("Displaying a random image from Radboud:")
display_image_and_mask(random_radboud['image_id'], random_radboud['data_provider'])

print("Displaying a random image from Karolinska:")
display_image_and_mask(random_karolinska['image_id'], random_karolinska['data_provider'])

In [ ]:
# Define patch size
patch_size = 1000

# Define a threshold for ROI (e.g., 10% non-background pixels)
roi_threshold = 0.10

# Create patches
def get_patches(image, mask, patch_size, stride=1000):
    patches = []
    mask_patches = []
    height, width = image.shape[:2]

    for i in range(0, height, stride):
        for j in range(0, width, stride):
            # Extract patch from image and mask
            image_patch = image[i:i+patch_size, j:j+patch_size]
            mask_patch = mask[i:i+patch_size, j:j+patch_size]

            # Check if patch size is valid (i.e., not smaller at edges)
            if image_patch.shape[0] == patch_size and image_patch.shape[1] == patch_size:
                patches.append(image_patch)
                mask_patches.append(mask_patch)

    return patches, mask_patches

# ROI Selection
def select_patches_with_roi(patches, mask_patches, roi_threshold):
    selected_patches = []

    for i, mask_patch in enumerate(mask_patches):
        # Calculate the percentage of non-background pixels
        non_background_pixels = np.count_nonzero(mask_patch)
        total_pixels = mask_patch.size
        percentage_non_background = non_background_pixels / total_pixels

        # Select patch if it has enough ROI
        if percentage_non_background > roi_threshold:
            selected_patches.append(patches[i])

    return selected_patches

In [ ]:
# Define patch size
patch_size = 1000
stride = 1000

In [ ]:
# Define a threshold for ROI (e.g., 10% non-background pixels)
roi_threshold = 0.10

In [ ]:
# Create patches from image and mask
def get_patches(image, mask, patch_size, stride=1000):
    patches = []
    mask_patches = []
    height, width = image.shape[:2]

    for i in range(0, height, stride):
        for j in range(0, width, stride):
            # Extract patch from image and mask
            image_patch = image[i:i+patch_size, j:j+patch_size]
            mask_patch = mask[i:i+patch_size, j:j+patch_size]

            # Check if patch size is valid (i.e., not smaller at edges)
            if image_patch.shape[0] == patch_size and image_patch.shape[1] == patch_size:
                patches.append(image_patch)
                mask_patches.append(mask_patch)

    return patches, mask_patches

In [ ]:
# ROI Selection function
def select_patches_with_roi(patches, mask_patches, roi_threshold):
    selected_patches = []
    selected_mask_patches = []

    for i, mask_patch in enumerate(mask_patches):
        # Calculate the percentage of non-background pixels
        non_background_pixels = np.count_nonzero(mask_patch)
        total_pixels = mask_patch.size
        percentage_non_background = non_background_pixels / total_pixels

        # Select patch if it has enough ROI
        if percentage_non_background > roi_threshold:
            selected_patches.append(patches[i])
            selected_mask_patches.append(mask_patch)  # Save corresponding mask patch

    return selected_patches, selected_mask_patches

In [ ]:
def apply_color_mapping(mask_patch, data_provider):
    color_mapping = {}
    if data_provider == 'radboud':
        color_mapping = {
              0: [255, 255, 255],  # White for background/unknown
              1: [0, 255, 0],      # Dark green for stroma (connective tissue)
              2: [255, 192, 203],  # Light pink for benign epithelium
              3: [255, 0, 0],      # Red for cancerous epithelium (Gleason 3)
              4: [200, 0, 0],      # Darker red for cancerous epithelium (Gleason 4)
              5: [150, 0, 0]       # Even darker red for cancerous epithelium (Gleason 5)
        }
    elif data_provider == 'karolinska':
        color_mapping = {
            0: [255, 255, 255],  # White for background/unknown
            1: [0, 255, 0],      # Green for benign tissue (stroma and epithelium combined)
            2: [255, 0, 0]       # Red for cancerous tissue (stroma and epithelium combined)
        }

    # Create a blank RGB array for the color-coded mask
    color_mask = np.zeros((mask_patch.shape[0], mask_patch.shape[1], 3), dtype=np.uint8)

    # Apply color mapping to the first channel of the mask
    for value, color in color_mapping.items():
        color_mask[mask_patch[:, :, 0] == value] = color  # Assign color to pixels based on mask value

    return color_mask

In [ ]:
# Function to display patches and their corresponding mask patches
def display_patches(image_patches, mask_patches, data_provider, num_display=5):
    plt.figure(figsize=(20, 20))

    for i in range(min(num_display, len(image_patches))):
        plt.subplot(num_display, 2, 2*i + 1)
        plt.imshow(image_patches[i])
        plt.title(f'Image Patch {i + 1} for {data_provider}' )
        plt.axis('off')

        # Apply color mapping to the corresponding mask patch
        color_mask = apply_color_mapping(mask_patches[i], data_provider)

        plt.subplot(num_display, 2, 2*i + 2)
        plt.imshow(color_mask)
        plt.title(f'Color-Coded Mask Patch {i + 1} for {data_provider}')
        plt.axis('off')

    plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Function to apply different thresholding methods and return the results
def apply_thresholding_and_mser(patch):
    gray_patch = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)

    # 1. Binary Thresholding
    _, binary_thresh = cv2.threshold(gray_patch, 128, 255, cv2.THRESH_BINARY)

    # 2. Adaptive Mean Thresholding
    adaptive_mean_thresh = cv2.adaptiveThreshold(gray_patch, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                                 cv2.THRESH_BINARY, 11, 2)

    # 3. Adaptive Gaussian Thresholding
    adaptive_gaussian_thresh = cv2.adaptiveThreshold(gray_patch, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                                     cv2.THRESH_BINARY, 11, 2)

    # 4. Otsu's Thresholding
    _, otsu_thresh = cv2.threshold(gray_patch, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 5. MSER Detection
    mser = cv2.MSER_create()
    regions, _ = mser.detectRegions(gray_patch)
    mser_image = np.zeros_like(gray_patch)

    for region in regions:
        convex_hull = cv2.convexHull(region.reshape(-1, 1, 2))
        cv2.polylines(mser_image, [convex_hull], 1, 255, thickness=2)

    return {
        'binary': binary_thresh,
        'adaptive_mean': adaptive_mean_thresh,
        'adaptive_gaussian': adaptive_gaussian_thresh,
        'otsu': otsu_thresh,
        'mser': mser_image
    }

import random

# Function to display the patch and its thresholded images with MSER
def display_thresholded_images_and_mser(patches, num_patches_to_display=5):
    # If there are fewer patches than requested, display all of them
    if len(patches) < num_patches_to_display:
        num_patches_to_display = len(patches)

    # Randomly sample the specified number of patches
    random_patches = random.sample(patches, num_patches_to_display)

    for patch_idx, patch in enumerate(random_patches):
        # Apply thresholding and MSER
        thresholded_images = apply_thresholding_and_mser(patch)

        # Display the original patch and thresholded results
        fig, axes = plt.subplots(1, 6, figsize=(24, 5))
        fig.suptitle(f'Patch {patch_idx + 1}: Original, Thresholding, and MSER')

        # Display the original patch
        axes[0].imshow(patch)
        axes[0].set_title('Original Patch')

        # Binary Thresholding
        axes[1].imshow(thresholded_images['binary'], cmap='gray')
        axes[1].set_title('Simple Binary Threshold (>128)')

        # Adaptive Mean Thresholding
        axes[2].imshow(thresholded_images['adaptive_mean'], cmap='gray')
        axes[2].set_title('Adaptive Mean Threshold')

        # Adaptive Gaussian Thresholding
        axes[3].imshow(thresholded_images['adaptive_gaussian'], cmap='gray')
        axes[3].set_title('Adaptive Gaussian Threshold')

        # Otsu's Thresholding
        axes[4].imshow(thresholded_images['otsu'], cmap='gray')
        axes[4].set_title('Otsu Threshold')

        # MSER
        axes[5].imshow(thresholded_images['mser'], cmap='gray')
        axes[5].set_title('MSER')

        # Remove axis for all images
        for ax in axes:
            ax.axis('off')

        plt.show()

In [ ]:
# Function to process the selected images
def process_image(image_id, data_provider):
    # Construct paths to the image and mask
    image_path = os.path.join(train_images_folder, f'{image_id}.tiff')
    mask_path = os.path.join(train_masks_folder, f'{image_id}_mask.tiff')

    # Open the image and mask
    image = np.array(Image.open(image_path))
    mask = np.array(Image.open(mask_path))

    # Get patches
    image_patches, mask_patches = get_patches(image, mask, patch_size, stride)

    # Select patches with ROI
    selected_patches, selected_mask_patches = select_patches_with_roi(image_patches, mask_patches, roi_threshold)

    # Display selected patches and their corresponding mask patches
    display_patches(selected_patches, selected_mask_patches, data_provider)


    display_thresholded_images_and_mser(selected_patches)


In [ ]:
# Process one image for each data provider
process_image(random_radboud['image_id'], 'radboud')
process_image(random_karolinska['image_id'], 'karolinska')

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from cellpose import models

In [ ]:

def apply_mser_with_params(patch, delta, min_area, max_area):
    gray_patch = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)

    mser = cv2.MSER_create(_delta=delta, _min_area=min_area, _max_area=max_area)

    regions, _ = mser.detectRegions(gray_patch)
    mser_image = np.zeros_like(gray_patch)

    for region in regions:
        convex_hull = cv2.convexHull(region.reshape(-1, 1, 2))
        cv2.polylines(mser_image, [convex_hull], 1, 255, thickness=2)

    return mser_image

def apply_cellpose_with_params(patch, diameter, model_type='cyto'):
    gray_patch = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)

    # Load CellPose model
    model = models.Cellpose(gpu=False, model_type=model_type)

    # Run CellPose with the given diameter
    masks, flows, styles, diams = model.eval(gray_patch, diameter=diameter, channels=[0, 0])

    return masks

def display_results_with_varying_params(patch, mser_params, cellpose_diameters):
    fig, axes = plt.subplots(len(mser_params) + 1, len(cellpose_diameters) + 1, figsize=(20, 20))

    axes[0, 0].imshow(patch)
    axes[0, 0].set_title('Original Image')
    axes[0, 0].axis('off')

    # Loop through MSER parameters and CellPose diameters
    for i, (delta, min_area, max_area) in enumerate(mser_params):
        for j, diameter in enumerate(cellpose_diameters):
            if j == 0:  # Apply MSER only for first column
                mser_result = apply_mser_with_params(patch, delta, min_area, max_area)
                axes[i + 1, 0].imshow(mser_result, cmap='gray')
                axes[i + 1, 0].set_title(f'MSER (Δ={delta}, minA={min_area}, maxA={max_area})')
                axes[i + 1, 0].axis('off')

            # Apply CellPose with varying diameters
            cellpose_result = apply_cellpose_with_params(patch, diameter)
            axes[i + 1, j + 1].imshow(cellpose_result, cmap='gray')
            axes[i + 1, j + 1].set_title(f'CellPose (diameter={diameter})')
            axes[i + 1, j + 1].axis('off')

    plt.tight_layout()
    plt.show()


mser_params = [
    (5, 60, 1000),
    (10, 80, 1500),
    (15, 100, 2000)
]

cellpose_diameters = [30, 50, 70]

display_results_with_varying_params(patch, mser_params, cellpose_diameters)